# MI24 — Inventário Físico — Análise Exploratória

**Tabela:** `dev_procurement.corp_curated.tbl_ds_log_mi24`
**Transação SAP:** MI24 · **Domínio:** Inventario fisico — documentos e itens de contagem
**Filtro:** _(nenhum — tabela completa)_
**Colunas:** 48 · **Clustering declarado:** `num_inventario_fisico, cod_centro`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente. Não há widget, view temporária nem ordem obrigatória.

## Alerta de partida

O `DESCRIBE` marca **três** colunas como PK nos comentários:
`num_inventario_fisico` + `num_item_inventario_fisico` + `num_exercicio_fiscal`.

Mas o **clustering** declara apenas `num_inventario_fisico` + `cod_centro` — que não inclui
o item nem o exercício. A seção 4 confirma qual das duas realmente identifica a linha.

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados da tabela |
| 2 | Volumetria |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| 6 | Preenchimento de todas as colunas |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| **9** | **Flags booleanas** |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas |
| 13 | Códigos e zeros à esquerda |
| **14** | **Chaves normalizadas para join** |
| **15** | **Checksum de linha** |
| 16 | Amostra |
| 17 | Distribuição interna |
| 18 | Análises específicas do inventário |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 15.


## 1. Metadados da tabela

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_log_mi24;

In [0]:
-- Formato, tamanho e particoes
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_log_mi24;

In [0]:
-- Ultimas gravacoes
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_log_mi24 LIMIT 20;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*)                                    AS linhas,
       COUNT(DISTINCT num_inventario_fisico)       AS documentos_inventario,
       COUNT(DISTINCT cod_centro)                  AS centros,
       COUNT(DISTINCT cod_material)                AS materiais,
       COUNT(DISTINCT cod_deposito)                AS depositos,
       COUNT(DISTINCT num_exercicio_fiscal)        AS exercicios
FROM dev_procurement.corp_curated.tbl_ds_log_mi24;

## 3. Distribuição por centro

Base para escolher o cenário de teste: recorte com volume viável (10 mil a 300 mil linhas).

In [0]:
-- 3. DISTRIBUICAO POR CENTRO
SELECT cod_centro,
       COUNT(*)                                AS linhas,
       COUNT(DISTINCT num_inventario_fisico)   AS documentos,
       COUNT(DISTINCT cod_material)            AS materiais,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mi24), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY cod_centro
ORDER BY linhas DESC
LIMIT 50;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional multiplicando linhas.

**Ponto crítico desta tabela:** compare o resultado da chave de 3 campos (PK declarada nos
comentários) com o clustering de 2 campos. Se a de 3 for única e a de 2 não, o clustering
está incompleto.

In [0]:
-- GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_mi24),
g AS (
  SELECT 'num_inventario_fisico' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_inventario_fisico` FROM dev_procurement.corp_curated.tbl_ds_log_mi24)
UNION ALL
  SELECT 'num_inventario_fisico + num_item_inventario_fisico' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_inventario_fisico`, `num_item_inventario_fisico` FROM dev_procurement.corp_curated.tbl_ds_log_mi24)
UNION ALL
  SELECT 'num_inventario_fisico + num_item_inventario_fisico + num_exercicio_fiscal' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_inventario_fisico`, `num_item_inventario_fisico`, `num_exercicio_fiscal` FROM dev_procurement.corp_curated.tbl_ds_log_mi24)
UNION ALL
  SELECT 'num_inventario_fisico + num_item_inventario_fisico + num_exercicio_fiscal + dateingest' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_inventario_fisico`, `num_item_inventario_fisico`, `num_exercicio_fiscal`, `dateingest` FROM dev_procurement.corp_curated.tbl_ds_log_mi24)
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave de clustering `num_inventario_fisico + cod_centro`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `num_inventario_fisico`, `cod_centro`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY `num_inventario_fisico`, `cod_centro`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.tbl_ds_log_mi24
),
dup AS (
  SELECT `num_inventario_fisico`, `cod_centro` FROM cen GROUP BY `num_inventario_fisico`, `cod_centro` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`num_inventario_fisico` <=> dup.`num_inventario_fisico` AND c.`cod_centro` <=> dup.`cod_centro`
),
agg AS (
  SELECT `num_inventario_fisico`, `cod_centro`,
         COUNT(DISTINCT `num_item_inventario_fisico`) AS `num_item_inventario_fisico`,
         COUNT(DISTINCT `num_exercicio_fiscal`) AS `num_exercicio_fiscal`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `tp_categoria_material`) AS `tp_categoria_material`,
         COUNT(DISTINCT `num_lote`) AS `num_lote`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `ind_item_contado`) AS `ind_item_contado`,
         COUNT(DISTINCT `cod_diferenca_item_esta_registrada`) AS `cod_diferenca_item_esta_registrada`,
         COUNT(DISTINCT `ind_item_recontado`) AS `ind_item_recontado`,
         COUNT(DISTINCT `ind_item_eliminado`) AS `ind_item_eliminado`,
         COUNT(DISTINCT `cod_contagem_zero`) AS `cod_contagem_zero`,
         COUNT(DISTINCT `tp_estoque_inventario_fisico`) AS `tp_estoque_inventario_fisico`,
         COUNT(DISTINCT `tp_estoque_especial`) AS `tp_estoque_especial`,
         COUNT(DISTINCT `tp_inventario_rotativo`) AS `tp_inventario_rotativo`,
         COUNT(DISTINCT `vl_quantidade_contagem_inventario`) AS `vl_quantidade_contagem_inventario`,
         COUNT(DISTINCT `vl_diferenca_inventario_entrada_contagem`) AS `vl_diferenca_inventario_entrada_contagem`,
         COUNT(DISTINCT `vl_precos_venda_excluindo_iva`) AS `vl_precos_venda_excluindo_iva`,
         COUNT(DISTINCT `vl_precos_venda_iva`) AS `vl_precos_venda_iva`,
         COUNT(DISTINCT `vl_inserido_externamente_base_venda`) AS `vl_inserido_externamente_base_venda`,
         COUNT(DISTINCT `cod_cliente`) AS `cod_cliente`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `cod_ordem_cliente`) AS `cod_ordem_cliente`,
         COUNT(DISTINCT `num_item_pedido_venda`) AS `num_item_pedido_venda`,
         COUNT(DISTINCT `num_divisao_remessa`) AS `num_divisao_remessa`,
         COUNT(DISTINCT `ind_material_valor`) AS `ind_material_valor`,
         COUNT(DISTINCT `qt_registrada_antes_contagem`) AS `qt_registrada_antes_contagem`,
         COUNT(DISTINCT `qt_quantidade`) AS `qt_quantidade`,
         COUNT(DISTINCT `sg_medida_basica_material`) AS `sg_medida_basica_material`,
         COUNT(DISTINCT `vl_contabil`) AS `vl_contabil`,
         COUNT(DISTINCT `vl_diferenca`) AS `vl_diferenca`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `dt_lancamento`) AS `dt_lancamento`,
         COUNT(DISTINCT `dt_realizacao_ultima_contagem`) AS `dt_realizacao_ultima_contagem`,
         COUNT(DISTINCT `dt_ultima_modificacao`) AS `dt_ultima_modificacao`,
         COUNT(DISTINCT `cod_usuario_contagem`) AS `cod_usuario_contagem`,
         COUNT(DISTINCT `cod_modificado`) AS `cod_modificado`,
         COUNT(DISTINCT `num_referencia_inventario_fisico`) AS `num_referencia_inventario_fisico`,
         COUNT(DISTINCT `num_material`) AS `num_material`,
         COUNT(DISTINCT `num_ano_material`) AS `num_ano_material`,
         COUNT(DISTINCT `num_item_material`) AS `num_item_material`,
         COUNT(DISTINCT `num_recontagem`) AS `num_recontagem`,
         COUNT(DISTINCT `cod_motivo_diferenca`) AS `cod_motivo_diferenca`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`,
         COUNT(DISTINCT `yearingest`) AS `yearingest`,
         COUNT(DISTINCT `monthingest`) AS `monthingest`
  FROM d GROUP BY `num_inventario_fisico`, `cod_centro`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(46,
    'num_item_inventario_fisico', MAX(`num_item_inventario_fisico`),
    'num_exercicio_fiscal', MAX(`num_exercicio_fiscal`),
    'cod_material', MAX(`cod_material`),
    'desc_material', MAX(`desc_material`),
    'tp_categoria_material', MAX(`tp_categoria_material`),
    'num_lote', MAX(`num_lote`),
    'cod_deposito', MAX(`cod_deposito`),
    'ind_item_contado', MAX(`ind_item_contado`),
    'cod_diferenca_item_esta_registrada', MAX(`cod_diferenca_item_esta_registrada`),
    'ind_item_recontado', MAX(`ind_item_recontado`),
    'ind_item_eliminado', MAX(`ind_item_eliminado`),
    'cod_contagem_zero', MAX(`cod_contagem_zero`),
    'tp_estoque_inventario_fisico', MAX(`tp_estoque_inventario_fisico`),
    'tp_estoque_especial', MAX(`tp_estoque_especial`),
    'tp_inventario_rotativo', MAX(`tp_inventario_rotativo`),
    'vl_quantidade_contagem_inventario', MAX(`vl_quantidade_contagem_inventario`),
    'vl_diferenca_inventario_entrada_contagem', MAX(`vl_diferenca_inventario_entrada_contagem`),
    'vl_precos_venda_excluindo_iva', MAX(`vl_precos_venda_excluindo_iva`),
    'vl_precos_venda_iva', MAX(`vl_precos_venda_iva`),
    'vl_inserido_externamente_base_venda', MAX(`vl_inserido_externamente_base_venda`),
    'cod_cliente', MAX(`cod_cliente`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'cod_ordem_cliente', MAX(`cod_ordem_cliente`),
    'num_item_pedido_venda', MAX(`num_item_pedido_venda`),
    'num_divisao_remessa', MAX(`num_divisao_remessa`),
    'ind_material_valor', MAX(`ind_material_valor`),
    'qt_registrada_antes_contagem', MAX(`qt_registrada_antes_contagem`),
    'qt_quantidade', MAX(`qt_quantidade`),
    'sg_medida_basica_material', MAX(`sg_medida_basica_material`),
    'vl_contabil', MAX(`vl_contabil`),
    'vl_diferenca', MAX(`vl_diferenca`),
    'cod_moeda', MAX(`cod_moeda`),
    'dt_lancamento', MAX(`dt_lancamento`),
    'dt_realizacao_ultima_contagem', MAX(`dt_realizacao_ultima_contagem`),
    'dt_ultima_modificacao', MAX(`dt_ultima_modificacao`),
    'cod_usuario_contagem', MAX(`cod_usuario_contagem`),
    'cod_modificado', MAX(`cod_modificado`),
    'num_referencia_inventario_fisico', MAX(`num_referencia_inventario_fisico`),
    'num_material', MAX(`num_material`),
    'num_ano_material', MAX(`num_ano_material`),
    'num_item_material', MAX(`num_item_material`),
    'num_recontagem', MAX(`num_recontagem`),
    'cod_motivo_diferenca', MAX(`cod_motivo_diferenca`),
    'dateingest', MAX(`dateingest`),
    'yearingest', MAX(`yearingest`),
    'monthingest', MAX(`monthingest`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada.

Uma coluna pode ter dado na tabela toda e estar vazia no recorte — ou o contrário.

> Para as 6 colunas `boolean`, a coluna `zeros_ou_false` conta os valores `FALSE`.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS
-- Para boolean, a coluna 'zeros' conta os valores FALSE.
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_mi24),
perf AS (
  SELECT stack(48,
    'num_inventario_fisico', 'string', COUNT_IF(`num_inventario_fisico` IS NULL), COUNT_IF(`num_inventario_fisico` IS NOT NULL AND lower(trim(`num_inventario_fisico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_inventario_fisico`) RLIKE '^0+([.,]0+)?$'),
    'num_item_inventario_fisico', 'string', COUNT_IF(`num_item_inventario_fisico` IS NULL), COUNT_IF(`num_item_inventario_fisico` IS NOT NULL AND lower(trim(`num_item_inventario_fisico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_inventario_fisico`) RLIKE '^0+([.,]0+)?$'),
    'num_exercicio_fiscal', 'string', COUNT_IF(`num_exercicio_fiscal` IS NULL), COUNT_IF(`num_exercicio_fiscal` IS NOT NULL AND lower(trim(`num_exercicio_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_exercicio_fiscal`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'tp_categoria_material', 'string', COUNT_IF(`tp_categoria_material` IS NULL), COUNT_IF(`tp_categoria_material` IS NOT NULL AND lower(trim(`tp_categoria_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_categoria_material`) RLIKE '^0+([.,]0+)?$'),
    'num_lote', 'string', COUNT_IF(`num_lote` IS NULL), COUNT_IF(`num_lote` IS NOT NULL AND lower(trim(`num_lote`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_lote`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_contado', 'boolean', COUNT_IF(`ind_item_contado` IS NULL), 0L, COUNT_IF(`ind_item_contado` = false),
    'cod_diferenca_item_esta_registrada', 'boolean', COUNT_IF(`cod_diferenca_item_esta_registrada` IS NULL), 0L, COUNT_IF(`cod_diferenca_item_esta_registrada` = false),
    'ind_item_recontado', 'boolean', COUNT_IF(`ind_item_recontado` IS NULL), 0L, COUNT_IF(`ind_item_recontado` = false),
    'ind_item_eliminado', 'boolean', COUNT_IF(`ind_item_eliminado` IS NULL), 0L, COUNT_IF(`ind_item_eliminado` = false),
    'cod_contagem_zero', 'boolean', COUNT_IF(`cod_contagem_zero` IS NULL), 0L, COUNT_IF(`cod_contagem_zero` = false),
    'tp_estoque_inventario_fisico', 'string', COUNT_IF(`tp_estoque_inventario_fisico` IS NULL), COUNT_IF(`tp_estoque_inventario_fisico` IS NOT NULL AND lower(trim(`tp_estoque_inventario_fisico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_estoque_inventario_fisico`) RLIKE '^0+([.,]0+)?$'),
    'tp_estoque_especial', 'string', COUNT_IF(`tp_estoque_especial` IS NULL), COUNT_IF(`tp_estoque_especial` IS NOT NULL AND lower(trim(`tp_estoque_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_estoque_especial`) RLIKE '^0+([.,]0+)?$'),
    'tp_inventario_rotativo', 'string', COUNT_IF(`tp_inventario_rotativo` IS NULL), COUNT_IF(`tp_inventario_rotativo` IS NOT NULL AND lower(trim(`tp_inventario_rotativo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_inventario_rotativo`) RLIKE '^0+([.,]0+)?$'),
    'vl_quantidade_contagem_inventario', 'double', COUNT_IF(`vl_quantidade_contagem_inventario` IS NULL), 0L, COUNT_IF(`vl_quantidade_contagem_inventario` = 0),
    'vl_diferenca_inventario_entrada_contagem', 'double', COUNT_IF(`vl_diferenca_inventario_entrada_contagem` IS NULL), 0L, COUNT_IF(`vl_diferenca_inventario_entrada_contagem` = 0),
    'vl_precos_venda_excluindo_iva', 'double', COUNT_IF(`vl_precos_venda_excluindo_iva` IS NULL), 0L, COUNT_IF(`vl_precos_venda_excluindo_iva` = 0),
    'vl_precos_venda_iva', 'double', COUNT_IF(`vl_precos_venda_iva` IS NULL), 0L, COUNT_IF(`vl_precos_venda_iva` = 0),
    'vl_inserido_externamente_base_venda', 'double', COUNT_IF(`vl_inserido_externamente_base_venda` IS NULL), 0L, COUNT_IF(`vl_inserido_externamente_base_venda` = 0),
    'cod_cliente', 'string', COUNT_IF(`cod_cliente` IS NULL), COUNT_IF(`cod_cliente` IS NOT NULL AND lower(trim(`cod_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_cliente`) RLIKE '^0+([.,]0+)?$'),
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'cod_ordem_cliente', 'string', COUNT_IF(`cod_ordem_cliente` IS NULL), COUNT_IF(`cod_ordem_cliente` IS NOT NULL AND lower(trim(`cod_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ordem_cliente`) RLIKE '^0+([.,]0+)?$'),
    'num_item_pedido_venda', 'string', COUNT_IF(`num_item_pedido_venda` IS NULL), COUNT_IF(`num_item_pedido_venda` IS NOT NULL AND lower(trim(`num_item_pedido_venda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_pedido_venda`) RLIKE '^0+([.,]0+)?$'),
    'num_divisao_remessa', 'string', COUNT_IF(`num_divisao_remessa` IS NULL), COUNT_IF(`num_divisao_remessa` IS NOT NULL AND lower(trim(`num_divisao_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_divisao_remessa`) RLIKE '^0+([.,]0+)?$'),
    'ind_material_valor', 'boolean', COUNT_IF(`ind_material_valor` IS NULL), 0L, COUNT_IF(`ind_material_valor` = false),
    'qt_registrada_antes_contagem', 'decimal(13,3)', COUNT_IF(`qt_registrada_antes_contagem` IS NULL), 0L, COUNT_IF(`qt_registrada_antes_contagem` = 0),
    'qt_quantidade', 'decimal(13,3)', COUNT_IF(`qt_quantidade` IS NULL), 0L, COUNT_IF(`qt_quantidade` = 0),
    'sg_medida_basica_material', 'string', COUNT_IF(`sg_medida_basica_material` IS NULL), COUNT_IF(`sg_medida_basica_material` IS NOT NULL AND lower(trim(`sg_medida_basica_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_medida_basica_material`) RLIKE '^0+([.,]0+)?$'),
    'vl_contabil', 'double', COUNT_IF(`vl_contabil` IS NULL), 0L, COUNT_IF(`vl_contabil` = 0),
    'vl_diferenca', 'double', COUNT_IF(`vl_diferenca` IS NULL), 0L, COUNT_IF(`vl_diferenca` = 0),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'dt_lancamento', 'string', COUNT_IF(`dt_lancamento` IS NULL), COUNT_IF(`dt_lancamento` IS NOT NULL AND lower(trim(`dt_lancamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_lancamento`) RLIKE '^0+([.,]0+)?$'),
    'dt_realizacao_ultima_contagem', 'string', COUNT_IF(`dt_realizacao_ultima_contagem` IS NULL), COUNT_IF(`dt_realizacao_ultima_contagem` IS NOT NULL AND lower(trim(`dt_realizacao_ultima_contagem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_realizacao_ultima_contagem`) RLIKE '^0+([.,]0+)?$'),
    'dt_ultima_modificacao', 'string', COUNT_IF(`dt_ultima_modificacao` IS NULL), COUNT_IF(`dt_ultima_modificacao` IS NOT NULL AND lower(trim(`dt_ultima_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_usuario_contagem', 'string', COUNT_IF(`cod_usuario_contagem` IS NULL), COUNT_IF(`cod_usuario_contagem` IS NOT NULL AND lower(trim(`cod_usuario_contagem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_usuario_contagem`) RLIKE '^0+([.,]0+)?$'),
    'cod_modificado', 'string', COUNT_IF(`cod_modificado` IS NULL), COUNT_IF(`cod_modificado` IS NOT NULL AND lower(trim(`cod_modificado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_modificado`) RLIKE '^0+([.,]0+)?$'),
    'num_referencia_inventario_fisico', 'string', COUNT_IF(`num_referencia_inventario_fisico` IS NULL), COUNT_IF(`num_referencia_inventario_fisico` IS NOT NULL AND lower(trim(`num_referencia_inventario_fisico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_referencia_inventario_fisico`) RLIKE '^0+([.,]0+)?$'),
    'num_material', 'string', COUNT_IF(`num_material` IS NULL), COUNT_IF(`num_material` IS NOT NULL AND lower(trim(`num_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_material`) RLIKE '^0+([.,]0+)?$'),
    'num_ano_material', 'string', COUNT_IF(`num_ano_material` IS NULL), COUNT_IF(`num_ano_material` IS NOT NULL AND lower(trim(`num_ano_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ano_material`) RLIKE '^0+([.,]0+)?$'),
    'num_item_material', 'string', COUNT_IF(`num_item_material` IS NULL), COUNT_IF(`num_item_material` IS NOT NULL AND lower(trim(`num_item_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_material`) RLIKE '^0+([.,]0+)?$'),
    'num_recontagem', 'string', COUNT_IF(`num_recontagem` IS NULL), COUNT_IF(`num_recontagem` IS NOT NULL AND lower(trim(`num_recontagem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_recontagem`) RLIKE '^0+([.,]0+)?$'),
    'cod_motivo_diferenca', 'string', COUNT_IF(`cod_motivo_diferenca` IS NULL), COUNT_IF(`cod_motivo_diferenca` IS NOT NULL AND lower(trim(`cod_motivo_diferenca`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_motivo_diferenca`) RLIKE '^0+([.,]0+)?$'),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L,
    'yearingest', 'string', COUNT_IF(`yearingest` IS NULL), COUNT_IF(`yearingest` IS NOT NULL AND lower(trim(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`yearingest`) RLIKE '^0+([.,]0+)?$'),
    'monthingest', 'string', COUNT_IF(`monthingest` IS NULL), COUNT_IF(`monthingest` IS NOT NULL AND lower(trim(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`monthingest`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                                THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0              THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01  THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                       AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

In [0]:
-- CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_mi24),
card AS (
  SELECT stack(48,
    'num_inventario_fisico', 'string', approx_count_distinct(`num_inventario_fisico`),
    'num_item_inventario_fisico', 'string', approx_count_distinct(`num_item_inventario_fisico`),
    'num_exercicio_fiscal', 'string', approx_count_distinct(`num_exercicio_fiscal`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'tp_categoria_material', 'string', approx_count_distinct(`tp_categoria_material`),
    'num_lote', 'string', approx_count_distinct(`num_lote`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'ind_item_contado', 'boolean', approx_count_distinct(`ind_item_contado`),
    'cod_diferenca_item_esta_registrada', 'boolean', approx_count_distinct(`cod_diferenca_item_esta_registrada`),
    'ind_item_recontado', 'boolean', approx_count_distinct(`ind_item_recontado`),
    'ind_item_eliminado', 'boolean', approx_count_distinct(`ind_item_eliminado`),
    'cod_contagem_zero', 'boolean', approx_count_distinct(`cod_contagem_zero`),
    'tp_estoque_inventario_fisico', 'string', approx_count_distinct(`tp_estoque_inventario_fisico`),
    'tp_estoque_especial', 'string', approx_count_distinct(`tp_estoque_especial`),
    'tp_inventario_rotativo', 'string', approx_count_distinct(`tp_inventario_rotativo`),
    'vl_quantidade_contagem_inventario', 'double', approx_count_distinct(`vl_quantidade_contagem_inventario`),
    'vl_diferenca_inventario_entrada_contagem', 'double', approx_count_distinct(`vl_diferenca_inventario_entrada_contagem`),
    'vl_precos_venda_excluindo_iva', 'double', approx_count_distinct(`vl_precos_venda_excluindo_iva`),
    'vl_precos_venda_iva', 'double', approx_count_distinct(`vl_precos_venda_iva`),
    'vl_inserido_externamente_base_venda', 'double', approx_count_distinct(`vl_inserido_externamente_base_venda`),
    'cod_cliente', 'string', approx_count_distinct(`cod_cliente`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'cod_ordem_cliente', 'string', approx_count_distinct(`cod_ordem_cliente`),
    'num_item_pedido_venda', 'string', approx_count_distinct(`num_item_pedido_venda`),
    'num_divisao_remessa', 'string', approx_count_distinct(`num_divisao_remessa`),
    'ind_material_valor', 'boolean', approx_count_distinct(`ind_material_valor`),
    'qt_registrada_antes_contagem', 'decimal(13,3)', approx_count_distinct(`qt_registrada_antes_contagem`),
    'qt_quantidade', 'decimal(13,3)', approx_count_distinct(`qt_quantidade`),
    'sg_medida_basica_material', 'string', approx_count_distinct(`sg_medida_basica_material`),
    'vl_contabil', 'double', approx_count_distinct(`vl_contabil`),
    'vl_diferenca', 'double', approx_count_distinct(`vl_diferenca`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'dt_lancamento', 'string', approx_count_distinct(`dt_lancamento`),
    'dt_realizacao_ultima_contagem', 'string', approx_count_distinct(`dt_realizacao_ultima_contagem`),
    'dt_ultima_modificacao', 'string', approx_count_distinct(`dt_ultima_modificacao`),
    'cod_usuario_contagem', 'string', approx_count_distinct(`cod_usuario_contagem`),
    'cod_modificado', 'string', approx_count_distinct(`cod_modificado`),
    'num_referencia_inventario_fisico', 'string', approx_count_distinct(`num_referencia_inventario_fisico`),
    'num_material', 'string', approx_count_distinct(`num_material`),
    'num_ano_material', 'string', approx_count_distinct(`num_ano_material`),
    'num_item_material', 'string', approx_count_distinct(`num_item_material`),
    'num_recontagem', 'string', approx_count_distinct(`num_recontagem`),
    'cod_motivo_diferenca', 'string', approx_count_distinct(`cod_motivo_diferenca`),
    'dateingest', 'date', approx_count_distinct(`dateingest`),
    'yearingest', 'string', approx_count_distinct(`yearingest`),
    'monthingest', 'string', approx_count_distinct(`monthingest`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

In [0]:
-- DOMINIO DAS CATEGORICAS
(SELECT 'tp_categoria_material' AS coluna, CAST(`tp_categoria_material` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `tp_categoria_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_estoque_inventario_fisico' AS coluna, CAST(`tp_estoque_inventario_fisico` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `tp_estoque_inventario_fisico` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_estoque_especial' AS coluna, CAST(`tp_estoque_especial` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `tp_estoque_especial` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_inventario_rotativo' AS coluna, CAST(`tp_inventario_rotativo` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `tp_inventario_rotativo` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_medida_basica_material' AS coluna, CAST(`sg_medida_basica_material` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `sg_medida_basica_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_motivo_diferenca' AS coluna, CAST(`cod_motivo_diferenca` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `cod_motivo_diferenca` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'num_exercicio_fiscal' AS coluna, CAST(`num_exercicio_fiscal` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `num_exercicio_fiscal` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_usuario_contagem' AS coluna, CAST(`cod_usuario_contagem` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_mi24 GROUP BY `cod_usuario_contagem` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Flags booleanas

As 6 flags descrevem o ciclo de vida do item de inventário. Flag com um único valor
é candidata a nunca ter sido carregada — confira no SAP antes de concluir.

In [0]:
-- FLAGS BOOLEANAS
SELECT 'ind_item_contado' AS flag,
       COUNT_IF(`ind_item_contado` = true)  AS verdadeiro,
       COUNT_IF(`ind_item_contado` = false) AS falso,
       COUNT_IF(`ind_item_contado` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_item_contado` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
UNION ALL
SELECT 'cod_diferenca_item_esta_registrada' AS flag,
       COUNT_IF(`cod_diferenca_item_esta_registrada` = true)  AS verdadeiro,
       COUNT_IF(`cod_diferenca_item_esta_registrada` = false) AS falso,
       COUNT_IF(`cod_diferenca_item_esta_registrada` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`cod_diferenca_item_esta_registrada` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
UNION ALL
SELECT 'ind_item_recontado' AS flag,
       COUNT_IF(`ind_item_recontado` = true)  AS verdadeiro,
       COUNT_IF(`ind_item_recontado` = false) AS falso,
       COUNT_IF(`ind_item_recontado` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_item_recontado` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
UNION ALL
SELECT 'ind_item_eliminado' AS flag,
       COUNT_IF(`ind_item_eliminado` = true)  AS verdadeiro,
       COUNT_IF(`ind_item_eliminado` = false) AS falso,
       COUNT_IF(`ind_item_eliminado` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_item_eliminado` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
UNION ALL
SELECT 'cod_contagem_zero' AS flag,
       COUNT_IF(`cod_contagem_zero` = true)  AS verdadeiro,
       COUNT_IF(`cod_contagem_zero` = false) AS falso,
       COUNT_IF(`cod_contagem_zero` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`cod_contagem_zero` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
UNION ALL
SELECT 'ind_material_valor' AS flag,
       COUNT_IF(`ind_material_valor` = true)  AS verdadeiro,
       COUNT_IF(`ind_material_valor` = false) AS falso,
       COUNT_IF(`ind_material_valor` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_material_valor` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
ORDER BY pct_verdadeiro DESC;

## 10. Perfil dos campos numéricos

Os 7 campos `double` exigem tolerância de 0,005 na comparação com o SAP.
Os 2 campos `decimal(13,3)` são seguros para conciliação exata.

In [0]:
-- PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(9,
    'vl_quantidade_contagem_inventario', 'double', COUNT(`vl_quantidade_contagem_inventario`), CAST(MIN(`vl_quantidade_contagem_inventario`) AS DOUBLE), CAST(MAX(`vl_quantidade_contagem_inventario`) AS DOUBLE), CAST(AVG(`vl_quantidade_contagem_inventario`) AS DOUBLE), CAST(percentile_approx(`vl_quantidade_contagem_inventario`, 0.5) AS DOUBLE), COUNT_IF(`vl_quantidade_contagem_inventario` < 0), COUNT_IF(`vl_quantidade_contagem_inventario` = 0),
    'vl_diferenca_inventario_entrada_contagem', 'double', COUNT(`vl_diferenca_inventario_entrada_contagem`), CAST(MIN(`vl_diferenca_inventario_entrada_contagem`) AS DOUBLE), CAST(MAX(`vl_diferenca_inventario_entrada_contagem`) AS DOUBLE), CAST(AVG(`vl_diferenca_inventario_entrada_contagem`) AS DOUBLE), CAST(percentile_approx(`vl_diferenca_inventario_entrada_contagem`, 0.5) AS DOUBLE), COUNT_IF(`vl_diferenca_inventario_entrada_contagem` < 0), COUNT_IF(`vl_diferenca_inventario_entrada_contagem` = 0),
    'vl_precos_venda_excluindo_iva', 'double', COUNT(`vl_precos_venda_excluindo_iva`), CAST(MIN(`vl_precos_venda_excluindo_iva`) AS DOUBLE), CAST(MAX(`vl_precos_venda_excluindo_iva`) AS DOUBLE), CAST(AVG(`vl_precos_venda_excluindo_iva`) AS DOUBLE), CAST(percentile_approx(`vl_precos_venda_excluindo_iva`, 0.5) AS DOUBLE), COUNT_IF(`vl_precos_venda_excluindo_iva` < 0), COUNT_IF(`vl_precos_venda_excluindo_iva` = 0),
    'vl_precos_venda_iva', 'double', COUNT(`vl_precos_venda_iva`), CAST(MIN(`vl_precos_venda_iva`) AS DOUBLE), CAST(MAX(`vl_precos_venda_iva`) AS DOUBLE), CAST(AVG(`vl_precos_venda_iva`) AS DOUBLE), CAST(percentile_approx(`vl_precos_venda_iva`, 0.5) AS DOUBLE), COUNT_IF(`vl_precos_venda_iva` < 0), COUNT_IF(`vl_precos_venda_iva` = 0),
    'vl_inserido_externamente_base_venda', 'double', COUNT(`vl_inserido_externamente_base_venda`), CAST(MIN(`vl_inserido_externamente_base_venda`) AS DOUBLE), CAST(MAX(`vl_inserido_externamente_base_venda`) AS DOUBLE), CAST(AVG(`vl_inserido_externamente_base_venda`) AS DOUBLE), CAST(percentile_approx(`vl_inserido_externamente_base_venda`, 0.5) AS DOUBLE), COUNT_IF(`vl_inserido_externamente_base_venda` < 0), COUNT_IF(`vl_inserido_externamente_base_venda` = 0),
    'qt_registrada_antes_contagem', 'decimal(13,3)', COUNT(`qt_registrada_antes_contagem`), CAST(MIN(`qt_registrada_antes_contagem`) AS DOUBLE), CAST(MAX(`qt_registrada_antes_contagem`) AS DOUBLE), CAST(AVG(`qt_registrada_antes_contagem`) AS DOUBLE), CAST(percentile_approx(`qt_registrada_antes_contagem`, 0.5) AS DOUBLE), COUNT_IF(`qt_registrada_antes_contagem` < 0), COUNT_IF(`qt_registrada_antes_contagem` = 0),
    'qt_quantidade', 'decimal(13,3)', COUNT(`qt_quantidade`), CAST(MIN(`qt_quantidade`) AS DOUBLE), CAST(MAX(`qt_quantidade`) AS DOUBLE), CAST(AVG(`qt_quantidade`) AS DOUBLE), CAST(percentile_approx(`qt_quantidade`, 0.5) AS DOUBLE), COUNT_IF(`qt_quantidade` < 0), COUNT_IF(`qt_quantidade` = 0),
    'vl_contabil', 'double', COUNT(`vl_contabil`), CAST(MIN(`vl_contabil`) AS DOUBLE), CAST(MAX(`vl_contabil`) AS DOUBLE), CAST(AVG(`vl_contabil`) AS DOUBLE), CAST(percentile_approx(`vl_contabil`, 0.5) AS DOUBLE), COUNT_IF(`vl_contabil` < 0), COUNT_IF(`vl_contabil` = 0),
    'vl_diferenca', 'double', COUNT(`vl_diferenca`), CAST(MIN(`vl_diferenca`) AS DOUBLE), CAST(MAX(`vl_diferenca`) AS DOUBLE), CAST(AVG(`vl_diferenca`) AS DOUBLE), CAST(percentile_approx(`vl_diferenca`, 0.5) AS DOUBLE), COUNT_IF(`vl_diferenca` < 0), COUNT_IF(`vl_diferenca` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(9,
    'vl_quantidade_contagem_inventario', CAST(SUM(`vl_quantidade_contagem_inventario`) AS DOUBLE), CAST(ROUND(SUM(`vl_quantidade_contagem_inventario`), 2) AS STRING), COUNT(`vl_quantidade_contagem_inventario`),
    'vl_diferenca_inventario_entrada_contagem', CAST(SUM(`vl_diferenca_inventario_entrada_contagem`) AS DOUBLE), CAST(ROUND(SUM(`vl_diferenca_inventario_entrada_contagem`), 2) AS STRING), COUNT(`vl_diferenca_inventario_entrada_contagem`),
    'vl_precos_venda_excluindo_iva', CAST(SUM(`vl_precos_venda_excluindo_iva`) AS DOUBLE), CAST(ROUND(SUM(`vl_precos_venda_excluindo_iva`), 2) AS STRING), COUNT(`vl_precos_venda_excluindo_iva`),
    'vl_precos_venda_iva', CAST(SUM(`vl_precos_venda_iva`) AS DOUBLE), CAST(ROUND(SUM(`vl_precos_venda_iva`), 2) AS STRING), COUNT(`vl_precos_venda_iva`),
    'vl_inserido_externamente_base_venda', CAST(SUM(`vl_inserido_externamente_base_venda`) AS DOUBLE), CAST(ROUND(SUM(`vl_inserido_externamente_base_venda`), 2) AS STRING), COUNT(`vl_inserido_externamente_base_venda`),
    'qt_registrada_antes_contagem', CAST(SUM(`qt_registrada_antes_contagem`) AS DOUBLE), CAST(ROUND(SUM(`qt_registrada_antes_contagem`), 2) AS STRING), COUNT(`qt_registrada_antes_contagem`),
    'qt_quantidade', CAST(SUM(`qt_quantidade`) AS DOUBLE), CAST(ROUND(SUM(`qt_quantidade`), 2) AS STRING), COUNT(`qt_quantidade`),
    'vl_contabil', CAST(SUM(`vl_contabil`) AS DOUBLE), CAST(ROUND(SUM(`vl_contabil`), 2) AS STRING), COUNT(`vl_contabil`),
    'vl_diferenca', CAST(SUM(`vl_diferenca`) AS DOUBLE), CAST(ROUND(SUM(`vl_diferenca`), 2) AS STRING), COUNT(`vl_diferenca`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
)
ORDER BY coluna;

## 12. Datas armazenadas como STRING

**Armadilha:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
Mesma data, formato diferente — normalizar antes de comparar.

In [0]:
-- DATAS EM STRING
SELECT coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo,
       CASE WHEN (CASE WHEN fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato' ELSE 'formato unico' END AS veredito
FROM (
  SELECT stack(3,
    'dt_lancamento', COUNT_IF(`dt_lancamento` IS NULL OR trim(`dt_lancamento`) = ''), COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_lancamento`) NOT IN ('', '00000000') THEN `dt_lancamento` END), MAX(CASE WHEN trim(`dt_lancamento`) NOT IN ('', '00000000') THEN `dt_lancamento` END),
    'dt_realizacao_ultima_contagem', COUNT_IF(`dt_realizacao_ultima_contagem` IS NULL OR trim(`dt_realizacao_ultima_contagem`) = ''), COUNT_IF(trim(`dt_realizacao_ultima_contagem`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_realizacao_ultima_contagem`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_realizacao_ultima_contagem`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_realizacao_ultima_contagem`) NOT IN ('', '00000000') THEN `dt_realizacao_ultima_contagem` END), MAX(CASE WHEN trim(`dt_realizacao_ultima_contagem`) NOT IN ('', '00000000') THEN `dt_realizacao_ultima_contagem` END),
    'dt_ultima_modificacao', COUNT_IF(`dt_ultima_modificacao` IS NULL OR trim(`dt_ultima_modificacao`) = ''), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END), MAX(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
)
ORDER BY coluna;

## 12.1 Datas em tipo nativo

In [0]:
-- DATAS NATIVAS
SELECT * FROM (
  SELECT stack(1,
    'dateingest', COUNT_IF(`dateingest` IS NULL), CAST(MIN(`dateingest`) AS STRING), CAST(MAX(`dateingest`) AS STRING), COUNT(DISTINCT `dateingest`), COUNT_IF(`dateingest` > current_date())
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(12,
    'num_inventario_fisico', 'string', COUNT_IF(CAST(`num_inventario_fisico` AS STRING) IS NULL OR trim(CAST(`num_inventario_fisico` AS STRING)) = ''), MIN(length(trim(CAST(`num_inventario_fisico` AS STRING)))), MAX(length(trim(CAST(`num_inventario_fisico` AS STRING)))), COUNT_IF(trim(CAST(`num_inventario_fisico` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_inventario_fisico` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_inventario_fisico` AS STRING)), '^0+', '')),
    'num_item_inventario_fisico', 'string', COUNT_IF(CAST(`num_item_inventario_fisico` AS STRING) IS NULL OR trim(CAST(`num_item_inventario_fisico` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_inventario_fisico` AS STRING)))), MAX(length(trim(CAST(`num_item_inventario_fisico` AS STRING)))), COUNT_IF(trim(CAST(`num_item_inventario_fisico` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_inventario_fisico` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_inventario_fisico` AS STRING)), '^0+', '')),
    'num_exercicio_fiscal', 'string', COUNT_IF(CAST(`num_exercicio_fiscal` AS STRING) IS NULL OR trim(CAST(`num_exercicio_fiscal` AS STRING)) = ''), MIN(length(trim(CAST(`num_exercicio_fiscal` AS STRING)))), MAX(length(trim(CAST(`num_exercicio_fiscal` AS STRING)))), COUNT_IF(trim(CAST(`num_exercicio_fiscal` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_exercicio_fiscal` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_exercicio_fiscal` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_deposito', 'string', COUNT_IF(CAST(`cod_deposito` AS STRING) IS NULL OR trim(CAST(`cod_deposito` AS STRING)) = ''), MIN(length(trim(CAST(`cod_deposito` AS STRING)))), MAX(length(trim(CAST(`cod_deposito` AS STRING)))), COUNT_IF(trim(CAST(`cod_deposito` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_deposito` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_deposito` AS STRING)), '^0+', '')),
    'num_lote', 'string', COUNT_IF(CAST(`num_lote` AS STRING) IS NULL OR trim(CAST(`num_lote` AS STRING)) = ''), MIN(length(trim(CAST(`num_lote` AS STRING)))), MAX(length(trim(CAST(`num_lote` AS STRING)))), COUNT_IF(trim(CAST(`num_lote` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_lote` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_lote` AS STRING)), '^0+', '')),
    'cod_cliente', 'string', COUNT_IF(CAST(`cod_cliente` AS STRING) IS NULL OR trim(CAST(`cod_cliente` AS STRING)) = ''), MIN(length(trim(CAST(`cod_cliente` AS STRING)))), MAX(length(trim(CAST(`cod_cliente` AS STRING)))), COUNT_IF(trim(CAST(`cod_cliente` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_cliente` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_cliente` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', '')),
    'cod_ordem_cliente', 'string', COUNT_IF(CAST(`cod_ordem_cliente` AS STRING) IS NULL OR trim(CAST(`cod_ordem_cliente` AS STRING)) = ''), MIN(length(trim(CAST(`cod_ordem_cliente` AS STRING)))), MAX(length(trim(CAST(`cod_ordem_cliente` AS STRING)))), COUNT_IF(trim(CAST(`cod_ordem_cliente` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_ordem_cliente` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_ordem_cliente` AS STRING)), '^0+', '')),
    'num_material', 'string', COUNT_IF(CAST(`num_material` AS STRING) IS NULL OR trim(CAST(`num_material` AS STRING)) = ''), MIN(length(trim(CAST(`num_material` AS STRING)))), MAX(length(trim(CAST(`num_material` AS STRING)))), COUNT_IF(trim(CAST(`num_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_material` AS STRING)), '^0+', '')),
    'num_recontagem', 'string', COUNT_IF(CAST(`num_recontagem` AS STRING) IS NULL OR trim(CAST(`num_recontagem` AS STRING)) = ''), MIN(length(trim(CAST(`num_recontagem` AS STRING)))), MAX(length(trim(CAST(`num_recontagem` AS STRING)))), COUNT_IF(trim(CAST(`num_recontagem` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_recontagem` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_recontagem` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
)
ORDER BY coluna;

## 14. Chaves normalizadas para join com o SAP

Chave completa (documento + item + exercício) já **sem zeros à esquerda**,
pronta para cruzar com o extrato do SAP via PROCV/ÍNDICE.

In [0]:
-- 14. CHAVES NORMALIZADAS
SELECT DISTINCT
       regexp_replace(trim(CAST(`num_inventario_fisico` AS STRING)), '^0+', '') AS `num_inventario_fisico_norm`,
       regexp_replace(trim(CAST(`num_item_inventario_fisico` AS STRING)), '^0+', '') AS `num_item_inventario_fisico_norm`,
       regexp_replace(trim(CAST(`num_exercicio_fiscal` AS STRING)), '^0+', '') AS `num_exercicio_fiscal_norm`
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
ORDER BY 1, 2, 3;

## 15. Checksum de linha

Impressão digital de cada linha. Se `linhas` > `linhas_unicas`, existem registros
100% idênticos — duplicata real, não granularidade.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                                AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_inventario_fisico` AS STRING), ''), COALESCE(CAST(`num_item_inventario_fisico` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`tp_categoria_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`ind_item_contado` AS STRING), ''), COALESCE(CAST(`cod_diferenca_item_esta_registrada` AS STRING), ''), COALESCE(CAST(`ind_item_recontado` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`cod_contagem_zero` AS STRING), ''), COALESCE(CAST(`tp_estoque_inventario_fisico` AS STRING), ''), COALESCE(CAST(`tp_estoque_especial` AS STRING), ''), COALESCE(CAST(`tp_inventario_rotativo` AS STRING), ''), COALESCE(CAST(`vl_quantidade_contagem_inventario` AS STRING), ''), COALESCE(CAST(`vl_diferenca_inventario_entrada_contagem` AS STRING), ''), COALESCE(CAST(`vl_precos_venda_excluindo_iva` AS STRING), ''), COALESCE(CAST(`vl_precos_venda_iva` AS STRING), ''), COALESCE(CAST(`vl_inserido_externamente_base_venda` AS STRING), ''), COALESCE(CAST(`cod_cliente` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_pedido_venda` AS STRING), ''), COALESCE(CAST(`num_divisao_remessa` AS STRING), ''), COALESCE(CAST(`ind_material_valor` AS STRING), ''), COALESCE(CAST(`qt_registrada_antes_contagem` AS STRING), ''), COALESCE(CAST(`qt_quantidade` AS STRING), ''), COALESCE(CAST(`sg_medida_basica_material` AS STRING), ''), COALESCE(CAST(`vl_contabil` AS STRING), ''), COALESCE(CAST(`vl_diferenca` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`dt_realizacao_ultima_contagem` AS STRING), ''), COALESCE(CAST(`dt_ultima_modificacao` AS STRING), ''), COALESCE(CAST(`cod_usuario_contagem` AS STRING), ''), COALESCE(CAST(`cod_modificado` AS STRING), ''), COALESCE(CAST(`num_referencia_inventario_fisico` AS STRING), ''), COALESCE(CAST(`num_material` AS STRING), ''), COALESCE(CAST(`num_ano_material` AS STRING), ''), COALESCE(CAST(`num_item_material` AS STRING), ''), COALESCE(CAST(`num_recontagem` AS STRING), ''), COALESCE(CAST(`cod_motivo_diferenca` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))                AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_inventario_fisico` AS STRING), ''), COALESCE(CAST(`num_item_inventario_fisico` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`tp_categoria_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`ind_item_contado` AS STRING), ''), COALESCE(CAST(`cod_diferenca_item_esta_registrada` AS STRING), ''), COALESCE(CAST(`ind_item_recontado` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`cod_contagem_zero` AS STRING), ''), COALESCE(CAST(`tp_estoque_inventario_fisico` AS STRING), ''), COALESCE(CAST(`tp_estoque_especial` AS STRING), ''), COALESCE(CAST(`tp_inventario_rotativo` AS STRING), ''), COALESCE(CAST(`vl_quantidade_contagem_inventario` AS STRING), ''), COALESCE(CAST(`vl_diferenca_inventario_entrada_contagem` AS STRING), ''), COALESCE(CAST(`vl_precos_venda_excluindo_iva` AS STRING), ''), COALESCE(CAST(`vl_precos_venda_iva` AS STRING), ''), COALESCE(CAST(`vl_inserido_externamente_base_venda` AS STRING), ''), COALESCE(CAST(`cod_cliente` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_pedido_venda` AS STRING), ''), COALESCE(CAST(`num_divisao_remessa` AS STRING), ''), COALESCE(CAST(`ind_material_valor` AS STRING), ''), COALESCE(CAST(`qt_registrada_antes_contagem` AS STRING), ''), COALESCE(CAST(`qt_quantidade` AS STRING), ''), COALESCE(CAST(`sg_medida_basica_material` AS STRING), ''), COALESCE(CAST(`vl_contabil` AS STRING), ''), COALESCE(CAST(`vl_diferenca` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`dt_realizacao_ultima_contagem` AS STRING), ''), COALESCE(CAST(`dt_ultima_modificacao` AS STRING), ''), COALESCE(CAST(`cod_usuario_contagem` AS STRING), ''), COALESCE(CAST(`cod_modificado` AS STRING), ''), COALESCE(CAST(`num_referencia_inventario_fisico` AS STRING), ''), COALESCE(CAST(`num_material` AS STRING), ''), COALESCE(CAST(`num_ano_material` AS STRING), ''), COALESCE(CAST(`num_item_material` AS STRING), ''), COALESCE(CAST(`num_recontagem` AS STRING), ''), COALESCE(CAST(`cod_motivo_diferenca` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))     AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_inventario_fisico` AS STRING), ''), COALESCE(CAST(`num_item_inventario_fisico` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`tp_categoria_material` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`ind_item_contado` AS STRING), ''), COALESCE(CAST(`cod_diferenca_item_esta_registrada` AS STRING), ''), COALESCE(CAST(`ind_item_recontado` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`cod_contagem_zero` AS STRING), ''), COALESCE(CAST(`tp_estoque_inventario_fisico` AS STRING), ''), COALESCE(CAST(`tp_estoque_especial` AS STRING), ''), COALESCE(CAST(`tp_inventario_rotativo` AS STRING), ''), COALESCE(CAST(`vl_quantidade_contagem_inventario` AS STRING), ''), COALESCE(CAST(`vl_diferenca_inventario_entrada_contagem` AS STRING), ''), COALESCE(CAST(`vl_precos_venda_excluindo_iva` AS STRING), ''), COALESCE(CAST(`vl_precos_venda_iva` AS STRING), ''), COALESCE(CAST(`vl_inserido_externamente_base_venda` AS STRING), ''), COALESCE(CAST(`cod_cliente` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_ordem_cliente` AS STRING), ''), COALESCE(CAST(`num_item_pedido_venda` AS STRING), ''), COALESCE(CAST(`num_divisao_remessa` AS STRING), ''), COALESCE(CAST(`ind_material_valor` AS STRING), ''), COALESCE(CAST(`qt_registrada_antes_contagem` AS STRING), ''), COALESCE(CAST(`qt_quantidade` AS STRING), ''), COALESCE(CAST(`sg_medida_basica_material` AS STRING), ''), COALESCE(CAST(`vl_contabil` AS STRING), ''), COALESCE(CAST(`vl_diferenca` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`dt_realizacao_ultima_contagem` AS STRING), ''), COALESCE(CAST(`dt_ultima_modificacao` AS STRING), ''), COALESCE(CAST(`cod_usuario_contagem` AS STRING), ''), COALESCE(CAST(`cod_modificado` AS STRING), ''), COALESCE(CAST(`num_referencia_inventario_fisico` AS STRING), ''), COALESCE(CAST(`num_material` AS STRING), ''), COALESCE(CAST(`num_ano_material` AS STRING), ''), COALESCE(CAST(`num_item_material` AS STRING), ''), COALESCE(CAST(`num_recontagem` AS STRING), ''), COALESCE(CAST(`cod_motivo_diferenca` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_log_mi24;

## 16. Amostra de linhas completas

In [0]:
-- 16. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_log_mi24
ORDER BY `num_inventario_fisico`, `num_item_inventario_fisico`
LIMIT 20;

## 17. Distribuição interna

In [0]:
-- 17. DISTRIBUICAO POR cod_deposito
SELECT COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)') AS `cod_deposito`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mi24), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR num_exercicio_fiscal
SELECT COALESCE(NULLIF(trim(CAST(`num_exercicio_fiscal` AS STRING)), ''), '(vazio)') AS `num_exercicio_fiscal`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mi24), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY COALESCE(NULLIF(trim(CAST(`num_exercicio_fiscal` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR tp_estoque_inventario_fisico
SELECT COALESCE(NULLIF(trim(CAST(`tp_estoque_inventario_fisico` AS STRING)), ''), '(vazio)') AS `tp_estoque_inventario_fisico`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mi24), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_estoque_inventario_fisico` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR cod_motivo_diferenca
SELECT COALESCE(NULLIF(trim(CAST(`cod_motivo_diferenca` AS STRING)), ''), '(vazio)') AS `cod_motivo_diferenca`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mi24), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_motivo_diferenca` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

### 17.1 Freshness

In [0]:
-- 17.1 FRESHNESS
SELECT dateingest AS data_carga,
       COUNT(*)   AS linhas,
       COUNT(DISTINCT num_inventario_fisico) AS documentos
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY dateingest
ORDER BY data_carga DESC
LIMIT 30;

In [0]:
-- 17.2 SNAPSHOT OU HISTORICO?
SELECT MIN(dateingest)                AS primeira_carga,
       MAX(dateingest)                AS ultima_carga,
       COUNT(DISTINCT dateingest)     AS cargas_distintas,
       CASE WHEN COUNT(DISTINCT dateingest) = 1
            THEN 'SNAPSHOT - substitui a cada carga; chave NAO precisa da data'
            ELSE 'HISTORICO - acumula; a chave DEVE incluir dateingest' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_log_mi24;

## 18. Análises específicas — MI24 (inventário físico)

### 18.1 Coerência da contagem

A MI24 registra a contagem física contra o saldo contábil. A relação esperada é:

`qt_quantidade` (contado) − `qt_registrada_antes_contagem` (saldo em livro) = diferença

Esta célula verifica se a diferença registrada bate com a diferença calculada.

In [0]:
-- 18.1 COERENCIA DA CONTAGEM
SELECT COUNT(*)                                                          AS linhas,
       COUNT_IF(qt_quantidade IS NULL)                                   AS sem_qtd_contada,
       COUNT_IF(qt_registrada_antes_contagem IS NULL)                    AS sem_qtd_livro,
       COUNT_IF(qt_quantidade = qt_registrada_antes_contagem)            AS contagem_bate,
       COUNT_IF(qt_quantidade > qt_registrada_antes_contagem)            AS sobra_fisica,
       COUNT_IF(qt_quantidade < qt_registrada_antes_contagem)            AS falta_fisica,
       ROUND(SUM(qt_quantidade - qt_registrada_antes_contagem), 3)       AS diferenca_qtd_total,
       ROUND(SUM(vl_diferenca), 2)                                       AS valor_diferenca_total
FROM dev_procurement.corp_curated.tbl_ds_log_mi24;

### 18.2 Diferença de quantidade × valor de diferença

Se a quantidade contada é igual ao saldo em livro, o valor de diferença deveria ser zero.
Divergência aqui indica erro de carga ou arredondamento.

In [0]:
-- 18.2 QUANTIDADE x VALOR DE DIFERENCA
SELECT COUNT(*) AS linhas,
       COUNT_IF(qt_quantidade = qt_registrada_antes_contagem
            AND ABS(COALESCE(vl_diferenca, 0)) > 0.005)  AS qtd_igual_mas_valor_diferente,
       COUNT_IF(qt_quantidade <> qt_registrada_antes_contagem
            AND ABS(COALESCE(vl_diferenca, 0)) <= 0.005) AS qtd_diferente_mas_valor_zero,
       COUNT_IF(vl_diferenca < 0)                        AS valor_diferenca_negativo,
       COUNT_IF(vl_contabil < 0)                         AS valor_contabil_negativo,
       CASE WHEN COUNT_IF(qt_quantidade = qt_registrada_antes_contagem
                      AND ABS(COALESCE(vl_diferenca, 0)) > 0.005) = 0
            THEN 'OK' ELSE 'ATENCAO - investigar' END    AS veredito
FROM dev_procurement.corp_curated.tbl_ds_log_mi24;

### 18.3 Matriz de status do item

As 6 flags booleanas descrevem o ciclo de vida do item de inventário.
Combinações impossíveis indicam problema de carga — por exemplo, item **não contado**
mas com **diferença registrada**.

In [0]:
-- 18.3 MATRIZ DE STATUS DO ITEM
SELECT ind_item_contado                    AS contado,
       cod_diferenca_item_esta_registrada  AS diferenca_registrada,
       ind_item_recontado                  AS recontado,
       ind_item_eliminado                  AS eliminado,
       cod_contagem_zero                   AS contagem_zero,
       COUNT(*)                            AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_mi24), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY ind_item_contado, cod_diferenca_item_esta_registrada,
         ind_item_recontado, ind_item_eliminado, cod_contagem_zero
ORDER BY linhas DESC;

In [0]:
-- 18.3b COMBINACOES LOGICAMENTE SUSPEITAS
SELECT 'nao contado MAS com diferenca registrada' AS situacao,
       COUNT_IF(ind_item_contado = false AND cod_diferenca_item_esta_registrada = true) AS linhas
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
UNION ALL
SELECT 'nao contado MAS com quantidade contada > 0',
       COUNT_IF(ind_item_contado = false AND qt_quantidade > 0)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
UNION ALL
SELECT 'contagem zero MAS quantidade contada <> 0',
       COUNT_IF(cod_contagem_zero = true AND qt_quantidade <> 0)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
UNION ALL
SELECT 'recontado MAS sem documento de recontagem',
       COUNT_IF(ind_item_recontado = true
            AND (num_recontagem IS NULL OR trim(num_recontagem) = ''))
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
UNION ALL
SELECT 'eliminado MAS com diferenca registrada',
       COUNT_IF(ind_item_eliminado = true AND cod_diferenca_item_esta_registrada = true)
  FROM dev_procurement.corp_curated.tbl_ds_log_mi24
ORDER BY linhas DESC;

### 18.4 Motivo da diferença

Preenchido apenas quando há divergência entre contagem e saldo em livro.

In [0]:
-- 18.4 MOTIVO DA DIFERENCA
SELECT COALESCE(NULLIF(trim(cod_motivo_diferenca), ''), '(vazio)') AS motivo,
       COUNT(*)                             AS linhas,
       COUNT_IF(qt_quantidade <> qt_registrada_antes_contagem) AS com_divergencia_qtd,
       ROUND(SUM(vl_diferenca), 2)          AS valor_diferenca
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY COALESCE(NULLIF(trim(cod_motivo_diferenca), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 30;

### 18.5 Itens por documento de inventário

Um documento de inventário agrupa vários itens. Esta distribuição ajuda a confirmar
se o extrato do SAP traz a mesma composição.

In [0]:
-- 18.5 ITENS POR DOCUMENTO
SELECT itens_no_documento, COUNT(*) AS documentos
FROM (SELECT num_inventario_fisico, num_exercicio_fiscal,
             COUNT(*) AS itens_no_documento
        FROM dev_procurement.corp_curated.tbl_ds_log_mi24
       GROUP BY num_inventario_fisico, num_exercicio_fiscal)
GROUP BY itens_no_documento
ORDER BY itens_no_documento
LIMIT 40;

### 18.6 Vínculo com o documento de material

Quando a diferença é registrada, o SAP gera um documento de material.
Esta célula verifica a consistência desse vínculo.

In [0]:
-- 18.6 VINCULO COM DOCUMENTO DE MATERIAL
SELECT COUNT(*) AS linhas,
       COUNT_IF(num_material IS NOT NULL AND trim(num_material) <> '')  AS com_doc_material,
       COUNT_IF(num_material IS NULL OR trim(num_material) = '')        AS sem_doc_material,
       COUNT_IF(cod_diferenca_item_esta_registrada = true
            AND (num_material IS NULL OR trim(num_material) = ''))      AS registrada_sem_doc,
       COUNT_IF(cod_diferenca_item_esta_registrada = false
            AND num_material IS NOT NULL AND trim(num_material) <> '')  AS doc_sem_registro,
       COUNT(DISTINCT num_material)                                     AS docs_distintos
FROM dev_procurement.corp_curated.tbl_ds_log_mi24;

### 18.7 Exercício fiscal e período de lançamento

Confirma a janela temporal do inventário — útil para alinhar o filtro da extração do SAP.

In [0]:
-- 18.7 EXERCICIO FISCAL x PERIODO
SELECT num_exercicio_fiscal,
       COUNT(*)                          AS linhas,
       COUNT(DISTINCT num_inventario_fisico) AS documentos,
       MIN(dt_lancamento)                AS lancamento_min,
       MAX(dt_lancamento)                AS lancamento_max,
       MIN(dt_realizacao_ultima_contagem) AS contagem_min,
       MAX(dt_realizacao_ultima_contagem) AS contagem_max
FROM dev_procurement.corp_curated.tbl_ds_log_mi24
GROUP BY num_exercicio_fiscal
ORDER BY num_exercicio_fiscal DESC;

## 19. Amostra ampliada

O notebook exploratório não extrai a base completa — use os notebooks de cenário para isso.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_log_mi24
ORDER BY `num_inventario_fisico`, `num_item_inventario_fisico`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_mi24),
g AS (
  SELECT 'num_inventario_fisico' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_inventario_fisico` FROM dev_procurement.corp_curated.tbl_ds_log_mi24)
UNION ALL
  SELECT 'num_inventario_fisico + num_item_inventario_fisico' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_inventario_fisico`, `num_item_inventario_fisico` FROM dev_procurement.corp_curated.tbl_ds_log_mi24)
UNION ALL
  SELECT 'num_inventario_fisico + num_item_inventario_fisico + num_exercicio_fiscal' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_inventario_fisico`, `num_item_inventario_fisico`, `num_exercicio_fiscal` FROM dev_procurement.corp_curated.tbl_ds_log_mi24)
UNION ALL
  SELECT 'num_inventario_fisico + num_item_inventario_fisico + num_exercicio_fiscal + dateingest' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_inventario_fisico`, `num_item_inventario_fisico`, `num_exercicio_fiscal`, `dateingest` FROM dev_procurement.corp_curated.tbl_ds_log_mi24)
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'MI24' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'dev_procurement.corp_curated.tbl_ds_log_mi24'
UNION ALL SELECT 'CENARIO', 'filtro', '(sem filtro)'
UNION ALL SELECT 'CENARIO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '48'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 3**.
2. Rodar os notebooks de cenário (`MI24_Centro4128_...` e `MI24_Centro4014_...`).
3. Extrair a MI24 no SAP com o mesmo recorte e na mesma data.

### Atenção especial nesta tabela

- **Chave:** confirmar se são 3 campos (PK dos comentários) ou 2 (clustering)
- **Booleanas:** o SAP exporta `X` / vazio; o Datalake grava `true` / `false`
- **Doubles:** 7 colunas de valor exigem tolerância de 0,005
